In [1]:
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, DataCollatorForSeq2Seq, TrainingArguments, Trainer

/root/miniconda3/envs/transformers/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
import transformers
print(transformers.__version__)

4.38.2


## 1、下载数据集

In [3]:
from datasets import load_dataset
ds = load_dataset("llamafactory/alpaca_zh", cache_dir="../data",split = "train")

In [4]:
ds

Dataset({
    features: ['instruction', 'input', 'output'],
    num_rows: 51155
})

In [5]:
ds = ds.train_test_split(test_size=0.8,seed = 42)
ds

DatasetDict({
    train: Dataset({
        features: ['instruction', 'input', 'output'],
        num_rows: 10231
    })
    test: Dataset({
        features: ['instruction', 'input', 'output'],
        num_rows: 40924
    })
})

In [6]:
ds['train'][5]

{'instruction': '解释气候变化对环境的两个影响',
 'input': '',
 'output': '气候变化对环境有几个影响。其中一个影响是全球平均气温上升，导致冰川融化、极端天气事件和海平面上升。另一个影响是由极端天气条件和升温引起的自然栖息地破坏导致的生物多样性减少。'}

## 2、数据预处理

In [7]:
#模型下载
# from modelscope import snapshot_download
# model_dir = snapshot_download('qwen/Qwen2-0.5B', cache_dir="./models")

In [8]:
model_path = "./models/qwen/Qwen2-0___5B"

In [9]:
tokenizer = AutoTokenizer.from_pretrained(model_path)
tokenizer

Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


Qwen2TokenizerFast(name_or_path='./models/qwen/Qwen2-0___5B', vocab_size=151643, model_max_length=32768, is_fast=True, padding_side='right', truncation_side='right', special_tokens={'eos_token': '<|endoftext|>', 'pad_token': '<|endoftext|>', 'additional_special_tokens': ['<|im_start|>', '<|im_end|>']}, clean_up_tokenization_spaces=False),  added_tokens_decoder={
	151643: AddedToken("<|endoftext|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	151644: AddedToken("<|im_start|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	151645: AddedToken("<|im_end|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
}

In [10]:
def process_func(datasets, tokenizer, max_length=256):
    """
    处理输入示例，合并用户的指令和输入字段，然后对输入和输出进行分词。
    该函数准备好训练自回归语言模型的数据格式。

    参数：
    - datasets (dict): 包含 'instruction'、'input' 和 'output' 字段的字典。
    - tokenizer (AutoTokenizer): 用于分词的 tokenizer。
    - max_length (int): 序列的最大长度。

    返回：
    - dict: 准备好的训练数据，包括 tokenized 的输入和标签。
    """
    # 1. 生成输入部分，包括 "Human: " 和 "Assistant: " 标签
    if datasets['input'].strip() == "":
        combined_input = "用户: " + datasets['instruction'] + "\n\n 机器人: "
    else:
        combined_input = "用户: " + datasets['instruction'] + "\n" + datasets['input'] + "\n\n 机器人: "

    # 2. 生成输出部分，并在最后加上终止标记
    output_text = datasets["output"] + tokenizer.eos_token  # 加上 eos_token 标记回复的结束

    # 3. 合并输入和输出作为模型的完整输入序列
    full_input = combined_input + output_text

    # 4. 对完整输入进行分词处理，确保生成 input_ids 和 attention_mask
    encodings = tokenizer(
        full_input, 
        max_length=max_length, 
        truncation=True, 
        padding="max_length", 
        return_tensors='pt'
    )

    # 5. 复制 input_ids 来生成标签
    labels = encodings["input_ids"].clone()

    # 6. 计算用户输入部分的长度（即需要忽略的部分）
    user_input_len = len(tokenizer(combined_input, truncation=True, max_length=max_length)["input_ids"])

    # 7. 将用户输入部分的标签设置为 -100，忽略这部分的损失
    labels[:, :user_input_len] = -100

    # 8. 返回 input_ids、attention_mask 和 labels，用于训练模型
    return {
        'input_ids': encodings['input_ids'].squeeze(0),  # 输入序列
        'attention_mask': encodings['attention_mask'].squeeze(0),  # 注意力掩码
        'labels': labels.squeeze(0)  # 标签，忽略用户输入部分
    }


In [11]:
# 假设 ds 是你的数据集，并且 tokenizer 已经定义
tokenized_ds = ds.map(
    process_func,
    remove_columns=['instruction', 'input', 'output'],  # 删除不需要的原始列
    fn_kwargs={'tokenizer': tokenizer},  # 将 tokenizer 传递给 process_func
)

tokenized_ds

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 10231
    })
    test: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 40924
    })
})

## 3、创建模型

In [12]:
model = AutoModelForCausalLM.from_pretrained(model_path)

### 3.1 设置微调参数

In [13]:
from peft import  AdaLoraConfig, TaskType, get_peft_model
config = AdaLoraConfig(task_type=TaskType.CAUSAL_LM) # init_r=12, lora_alpha=32, target_modules=["q", "v"],lora_dropout=0.01

### 3.2 重新载入模型

In [14]:
model = get_peft_model(model, config)
model

PeftModelForCausalLM(
  (base_model): AdaLoraModel(
    (model): Qwen2ForCausalLM(
      (model): Qwen2Model(
        (embed_tokens): Embedding(151936, 896)
        (layers): ModuleList(
          (0-23): 24 x Qwen2DecoderLayer(
            (self_attn): Qwen2SdpaAttention(
              (q_proj): adalora.SVDLinear(
                (base_layer): Linear(in_features=896, out_features=896, bias=True)
                (lora_dropout): ModuleDict(
                  (default): Identity()
                )
                (lora_A): ParameterDict(  (default): Parameter containing: [torch.FloatTensor of size 12x896])
                (lora_B): ParameterDict(  (default): Parameter containing: [torch.FloatTensor of size 896x12])
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
                (lora_E): ParameterDict(  (default): Parameter containing: [torch.FloatTensor of size 12x1])
         

In [15]:
config

AdaLoraConfig(peft_type=<PeftType.ADALORA: 'ADALORA'>, auto_mapping=None, base_model_name_or_path='./models/qwen/Qwen2-0___5B', revision=None, task_type=<TaskType.CAUSAL_LM: 'CAUSAL_LM'>, inference_mode=False, r=8, target_modules=['q_proj', 'v_proj'], lora_alpha=8, lora_dropout=0.0, fan_in_fan_out=False, bias='none', use_rslora=False, modules_to_save=None, init_lora_weights=True, layers_to_transform=None, layers_pattern=None, rank_pattern=None, alpha_pattern={}, megatron_config=None, megatron_core='megatron.core', loftq_config={}, use_dora=False, layer_replication=None, runtime_config=LoraRuntimeConfig(ephemeral_gpu_offload=False), target_r=8, init_r=12, tinit=0, tfinal=0, deltaT=1, beta1=0.85, beta2=0.85, orth_reg_weight=0.5, total_step=None)

In [16]:
for name, parameter in model.named_parameters():
    print(name)

base_model.model.model.embed_tokens.weight
base_model.model.model.layers.0.self_attn.q_proj.base_layer.weight
base_model.model.model.layers.0.self_attn.q_proj.base_layer.bias
base_model.model.model.layers.0.self_attn.q_proj.lora_A.default
base_model.model.model.layers.0.self_attn.q_proj.lora_B.default
base_model.model.model.layers.0.self_attn.q_proj.lora_E.default
base_model.model.model.layers.0.self_attn.q_proj.ranknum.default
base_model.model.model.layers.0.self_attn.k_proj.weight
base_model.model.model.layers.0.self_attn.k_proj.bias
base_model.model.model.layers.0.self_attn.v_proj.base_layer.weight
base_model.model.model.layers.0.self_attn.v_proj.base_layer.bias
base_model.model.model.layers.0.self_attn.v_proj.lora_A.default
base_model.model.model.layers.0.self_attn.v_proj.lora_B.default
base_model.model.model.layers.0.self_attn.v_proj.lora_E.default
base_model.model.model.layers.0.self_attn.v_proj.ranknum.default
base_model.model.model.layers.0.self_attn.o_proj.weight
base_model.mo

### 3.3 查看参数量

In [17]:
model.print_trainable_parameters()

trainable params: 811,584 || all params: 494,844,400 || trainable%: 0.1640


## 4、创建训练参数

In [18]:
args = TrainingArguments(
    output_dir="./models_for_AdaLoRA",
    per_device_train_batch_size=16,
    gradient_accumulation_steps=2,  # 每4步累计一次
    per_device_eval_batch_size=32,
    save_strategy="epoch",
    logging_steps=20,
    num_train_epochs=10,
    report_to=['tensorboard'],
)

## 5、创建训练器并训练

In [19]:
trainer = Trainer(
    model=model,
    args=args,
    train_dataset=tokenized_ds["train"], 
    eval_dataset=tokenized_ds["test"],     
    data_collator=DataCollatorForSeq2Seq(tokenizer=tokenizer, padding=True)
)

/root/miniconda3/envs/transformers/lib/python3.10/site-packages/accelerate/accelerator.py:457: FutureWarning: Passing the following arguments to `Accelerator` is deprecated and will be removed in version 1.0 of Accelerate: dict_keys(['dispatch_batches', 'split_batches', 'even_batches', 'use_seedable_sampler']). Please pass an `accelerate.DataLoaderConfiguration` instead: 
dataloader_config = DataLoaderConfiguration(dispatch_batches=None, split_batches=False, even_batches=True, use_seedable_sampler=True)
  warnings.warn(
Detected kernel version 3.10.0, which is below the recommended minimum of 5.5.0; this can cause the process to hang. It is recommended to upgrade the kernel to the minimum version or higher.


In [20]:
trainer.train()

Step,Training Loss
20,16.603100
40,16.731500
60,16.424000
80,16.464100
100,16.442900
120,16.071600
140,16.258200
160,15.822800
180,15.674900
200,15.557000


TrainOutput(global_step=3200, training_loss=3.3362471896409986, metrics={'train_runtime': 7271.16, 'train_samples_per_second': 14.071, 'train_steps_per_second': 0.44, 'total_flos': 5.637057624367104e+16, 'train_loss': 3.3362471896409986, 'epoch': 10.0})

## 6、预测

### 6.1 加载基础模型

In [21]:
from transformers import AutoTokenizer, AutoModelForCausalLM
model_path = "./models/qwen/Qwen2-0___5B"
model = AutoModelForCausalLM.from_pretrained(model_path)
tokenizer = AutoTokenizer.from_pretrained(model_path)

Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


### 6.2 加载LoRA训练模型参数

In [22]:
from peft import PeftModel
p_model = PeftModel.from_pretrained(model, model_id="./models_for_LoRA/checkpoint-319")
p_model

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): Qwen2ForCausalLM(
      (model): Qwen2Model(
        (embed_tokens): Embedding(151936, 896)
        (layers): ModuleList(
          (0-23): 24 x Qwen2DecoderLayer(
            (self_attn): Qwen2SdpaAttention(
              (q_proj): lora.Linear(
                (base_layer): Linear(in_features=896, out_features=896, bias=True)
                (lora_dropout): ModuleDict(
                  (default): Identity()
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=896, out_features=8, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=8, out_features=896, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj): Linear(in_features=896, out_feature

### 6.3 模型合并

In [23]:
merge_model = p_model.merge_and_unload()
merge_model

Qwen2ForCausalLM(
  (model): Qwen2Model(
    (embed_tokens): Embedding(151936, 896)
    (layers): ModuleList(
      (0-23): 24 x Qwen2DecoderLayer(
        (self_attn): Qwen2SdpaAttention(
          (q_proj): Linear(in_features=896, out_features=896, bias=True)
          (k_proj): Linear(in_features=896, out_features=128, bias=True)
          (v_proj): Linear(in_features=896, out_features=128, bias=True)
          (o_proj): Linear(in_features=896, out_features=896, bias=False)
          (rotary_emb): Qwen2RotaryEmbedding()
        )
        (mlp): Qwen2MLP(
          (gate_proj): Linear(in_features=896, out_features=4864, bias=False)
          (up_proj): Linear(in_features=896, out_features=4864, bias=False)
          (down_proj): Linear(in_features=4864, out_features=896, bias=False)
          (act_fn): SiLU()
        )
        (input_layernorm): Qwen2RMSNorm()
        (post_attention_layernorm): Qwen2RMSNorm()
      )
    )
    (norm): Qwen2RMSNorm()
  )
  (lm_head): Linear(in_featur

### 6.4 模型保存

In [24]:
merge_model.save_pretrained("./models/lora_merge_model")

### 6.5 加载

In [25]:
from transformers import  AutoModelForCausalLM
model_path_lora = "./models/lora_merge_model"
model_lora = AutoModelForCausalLM.from_pretrained(model_path_lora)
model_lora

Qwen2ForCausalLM(
  (model): Qwen2Model(
    (embed_tokens): Embedding(151936, 896)
    (layers): ModuleList(
      (0-23): 24 x Qwen2DecoderLayer(
        (self_attn): Qwen2SdpaAttention(
          (q_proj): Linear(in_features=896, out_features=896, bias=True)
          (k_proj): Linear(in_features=896, out_features=128, bias=True)
          (v_proj): Linear(in_features=896, out_features=128, bias=True)
          (o_proj): Linear(in_features=896, out_features=896, bias=False)
          (rotary_emb): Qwen2RotaryEmbedding()
        )
        (mlp): Qwen2MLP(
          (gate_proj): Linear(in_features=896, out_features=4864, bias=False)
          (up_proj): Linear(in_features=896, out_features=4864, bias=False)
          (down_proj): Linear(in_features=4864, out_features=896, bias=False)
          (act_fn): SiLU()
        )
        (input_layernorm): Qwen2RMSNorm()
        (post_attention_layernorm): Qwen2RMSNorm()
      )
    )
    (norm): Qwen2RMSNorm()
  )
  (lm_head): Linear(in_featur